<a href="https://colab.research.google.com/github/cse240840131002-ship-it/GTU_Internship/blob/main/Day_5(10_07_26).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OrdinalEncoder
# -------------------------------
# Function to generate data
# -------------------------------

def generate_student_data(num_samples=1000):

    stream = ['CS', 'Mech', 'EC', 'IT', 'Civil']
    genders = ['F', 'M']

    np.random.seed(42)

    data = pd.DataFrame({
        'Stream': np.random.choice(stream, num_samples),
        'Gender': np.random.choice(genders, num_samples)
    })

    data['Maths'] = np.round(np.clip(np.random.normal(6,1,num_samples),1,10))
    data['English'] = np.round(np.clip(np.random.normal(7,1,num_samples),1,10))
    data['Science'] = np.round(np.clip(np.random.normal(5,1,num_samples),1,10))

    data['Total'] = data['Maths'] + data['English'] + data['Science']

    def assign_grade(total):
        if total >= 24:
            return 'A'
        elif total >= 18:
            return 'B'
        elif total >= 15:
            return 'C'
        else:
            return 'D'

    data['Grade'] = data['Total'].apply(assign_grade)

    data['Fuel'] = np.round(
        np.clip(np.random.normal(10.5,3.16,num_samples),1,20),2
    )

    return data


# ------------------------------------
# Generate Original Dataset
# ------------------------------------

df = generate_student_data(1000)

print("Original Data")
print(df.head())

# ------------------------------------
# Create a Copy
# ------------------------------------

df_copy = df.copy()

# ------------------------------------
# One-Hot Encoding using get_dummies()
# ------------------------------------
print("\ncategorical")
df_copy = pd.get_dummies(
    df_copy,
    columns=['Stream', 'Gender'],
    dtype=int
)

# ------------------------------------
# Ordinal Encoding for Grade
# ------------------------------------

ordinal = OrdinalEncoder(categories=[['D','C','B','A']])

df_copy[['Grade']] = ordinal.fit_transform(df_copy[['Grade']])

# ------------------------------------
# Standard Scaling
# ------------------------------------

scaler = StandardScaler()

num_cols = ['Maths','English','Science','Total','Fuel']

df_copy[num_cols] = scaler.fit_transform(df_copy[num_cols])

# ------------------------------------
# Result
# ------------------------------------
print("\nordinal")
print("\nProcessed Data")
print(df_copy.head())

Original Data
  Stream Gender  Maths  English  Science  Total Grade   Fuel
0     IT      M    7.0      6.0      3.0   16.0     C   7.77
1  Civil      M    7.0      7.0      4.0   18.0     B  10.40
2     EC      M    6.0      6.0      5.0   17.0     C  10.56
3  Civil      M    5.0      7.0      7.0   19.0     B  11.99
4  Civil      M    7.0      5.0      6.0   18.0     B   6.18

categorical

ordinal

Processed Data
      Maths   English   Science     Total  Grade      Fuel  Stream_CS  \
0  0.882580 -0.971914 -1.872289 -1.146928    1.0 -0.822776          0   
1  0.882580 -0.002907 -0.921888 -0.024689    2.0  0.017665          0   
2 -0.067452 -0.971914  0.028512 -0.585809    1.0  0.068795          0   
3 -1.017484 -0.002907  1.929313  0.536430    2.0  0.525765          0   
4  0.882580 -1.940922  0.978912 -0.024689    2.0 -1.330875          0   

   Stream_Civil  Stream_EC  Stream_IT  Stream_Mech  Gender_F  Gender_M  
0             0          0          1            0         0         1

In [2]:
# ==========================
# Import Libraries
# ==========================

import pandas as pd
import numpy as np

from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler, LabelEncoder


In [7]:
# Load Dataset
# ============================================================

df = pd.read_csv("/content/renewable_energy(1).csv")
df

,Energy Source,Region,Year,Energy Production (GWh),Investment (Million $),CO2 Reduction (Tons),Jobs Created,Efficiency (%)
0,Wind,South America,2022,NaN,NaN,15834,259,58
1,Hydropower,Asia,2023,1224.0,204.0,13957,732,38
2,Geothermal,Europe,2021,1944.0,253.0,7103,562,53
3,Solar,Africa,2020,2779.0,529.0,12441,309,23
4,Biomass,Asia,2020,1343.0,359.0,13964,272,59
...,...,...,...,...,...,...,...,...
95,Hydropower,Asia,2022,1054.0,487.0,15146,247,28
96,Geothermal,Asia,2024,2392.0,218.0,6617,743,54
97,Wind,South America,2024,1667.0,260.0,19101,217,57
98,Biomass,Africa,2023,1750.0,582.0,4643,599,21


In [11]:
# Basic Information
# ============================================================

print("1.Information:-\n",df.info())

print("2.Describe data:-\n",df.describe())

print("3 Shape:-\n",df.shape)

print("4 Columns :-\n",df.columns)


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 8 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Energy Source            100 non-null    object 
 1   Region                   100 non-null    object 
 2   Year                     100 non-null    int64  
 3   Energy Production (GWh)  95 non-null     float64
 4   Investment (Million $)   94 non-null     float64
 5   CO2 Reduction (Tons)     100 non-null    int64  
 6   Jobs Created             100 non-null    int64  
 7   Efficiency (%)           100 non-null    int64  
dtypes: float64(2), int64(4), object(2)
memory usage: 6.4+ KB
1.Information:-
 None
2.Describe data:-
               Year  Energy Production (GWh)  Investment (Million $)  \
count   100.000000                95.000000               94.000000   
mean   2022.170000              1899.831579              376.500000   
std       1.775564               633.0071

In [12]:
# Check Missing Values
# ============================================================

print(df.isnull().sum())

Energy Source              0
Region                     0
Year                       0
Energy Production (GWh)    5
Investment (Million $)     6
CO2 Reduction (Tons)       0
Jobs Created               0
Efficiency (%)             0
dtype: int64


In [13]:
# Handle Missing Values
# ============================================================

# Numerical columns -> Mean
df["Energy Production (GWh)"] = df["Energy Production (GWh)"].fillna(
    df["Energy Production (GWh)"].mean())

df["Investment (Million $)"] = df["Investment (Million $)"].fillna(
    df["Investment (Million $)"].mean())

df["CO2 Reduction (Tons)"] = df["CO2 Reduction (Tons)"].fillna(
    df["CO2 Reduction (Tons)"].mean())

df["Jobs Created"] = df["Jobs Created"].fillna(
    df["Jobs Created"].mean())

df["Efficiency (%)"] = df["Efficiency (%)"].fillna(
    df["Efficiency (%)"].mean())


In [14]:
# Categorical -> Mode

df["Energy Source"] = df["Energy Source"].fillna(
    df["Energy Source"].mode()[0])

df["Region"] = df["Region"].fillna(
    df["Region"].mode()[0])

In [15]:
# Feature Engineering
# ============================================================

# Investment per GWh
df["Investment_per_GWh"] = df["Investment (Million $)"] / df["Energy Production (GWh)"]

# CO2 reduction per Job
df["CO2_per_Job"] = df["CO2 Reduction (Tons)"] / df["Jobs Created"]

In [16]:
# Generalisation
# ============================================================

# Average Energy Production by Source
print(df.groupby("Energy Source")["Energy Production (GWh)"].mean())

# Average Efficiency by Region
print(df.groupby("Region")["Efficiency (%)"].mean())

# Total Investment by Source
print(df.groupby("Energy Source")["Investment (Million $)"].sum())

Energy Source
Biomass       1992.541579
Geothermal    2084.872446
Hydropower    1840.191579
Solar         1936.862243
Wind          1666.891579
Name: Energy Production (GWh), dtype: float64
Region
Africa           38.222222
Asia             40.038462
Europe           41.500000
North America    39.952381
South America    41.789474
Name: Efficiency (%), dtype: float64
Energy Source
Biomass       8031.0
Geothermal    5328.0
Hydropower    7963.0
Solar         8800.5
Wind          7527.5
Name: Investment (Million $), dtype: float64


In [17]:
# Correlation
# ============================================================

print(df.corr(numeric_only=True))



                             Year  Energy Production (GWh)  \
Year                     1.000000                -0.097278   
Energy Production (GWh) -0.097278                 1.000000   
Investment (Million $)   0.042162                 0.100008   
CO2 Reduction (Tons)     0.092289                -0.025098   
Jobs Created             0.151308                -0.203983   
Efficiency (%)           0.169061                -0.210072   
Investment_per_GWh       0.115444                -0.643133   
CO2_per_Job             -0.069120                 0.081903   

                         Investment (Million $)  CO2 Reduction (Tons)  \
Year                                   0.042162              0.092289   
Energy Production (GWh)                0.100008             -0.025098   
Investment (Million $)                 1.000000              0.006550   
CO2 Reduction (Tons)                   0.006550              1.000000   
Jobs Created                          -0.208114             -0.074871   
Eff

In [21]:
# Label Encoding
# ============================================================

encoder = LabelEncoder()

df["Energy Source"] = encoder.fit_transform(df["Energy Source"])
df["Region"] = encoder.fit_transform(df["Region"])

df

,Energy Source,Region,Year,Energy Production (GWh),Investment (Million $),CO2 Reduction (Tons),Jobs Created,Efficiency (%),Investment_per_GWh,CO2_per_Job
0,4,4,2022,1899.831579,376.5,15834,259,58,0.198175,61.135135
1,2,1,2023,1224.000000,204.0,13957,732,38,0.166667,19.066940
2,1,2,2021,1944.000000,253.0,7103,562,53,0.130144,12.638790
3,3,0,2020,2779.000000,529.0,12441,309,23,0.190356,40.262136
4,0,1,2020,1343.000000,359.0,13964,272,59,0.267312,51.338235
...,...,...,...,...,...,...,...,...,...,...
95,2,1,2022,1054.000000,487.0,15146,247,28,0.462049,61.319838
96,1,1,2024,2392.000000,218.0,6617,743,54,0.091137,8.905787
97,4,4,2024,1667.000000,260.0,19101,217,57,0.155969,88.023041
98,0,0,2023,1750.000000,582.0,4643,599,21,0.332571,7.751252


In [20]:
# One Hot Encoding
# ============================================================

dummy_df = pd.get_dummies(df,
                          columns=["Energy Source","Region"])

df

,Energy Source,Region,Year,Energy Production (GWh),Investment (Million $),CO2 Reduction (Tons),Jobs Created,Efficiency (%),Investment_per_GWh,CO2_per_Job
0,4,4,2022,1899.831579,376.5,15834,259,58,0.198175,61.135135
1,2,1,2023,1224.000000,204.0,13957,732,38,0.166667,19.066940
2,1,2,2021,1944.000000,253.0,7103,562,53,0.130144,12.638790
3,3,0,2020,2779.000000,529.0,12441,309,23,0.190356,40.262136
4,0,1,2020,1343.000000,359.0,13964,272,59,0.267312,51.338235
...,...,...,...,...,...,...,...,...,...,...
95,2,1,2022,1054.000000,487.0,15146,247,28,0.462049,61.319838
96,1,1,2024,2392.000000,218.0,6617,743,54,0.091137,8.905787
97,4,4,2024,1667.000000,260.0,19101,217,57,0.155969,88.023041
98,0,0,2023,1750.000000,582.0,4643,599,21,0.332571,7.751252


In [22]:
scaler = StandardScaler()

numerical_columns = [
    "Energy Production (GWh)",
    "Investment (Million $)",
    "CO2 Reduction (Tons)",
    "Jobs Created",
    "Efficiency (%)",
    "Investment_per_GWh",
    "CO2_per_Job"
]

df[numerical_columns] = scaler.fit_transform(df[numerical_columns])

df


,Energy Source,Region,Year,Energy Production (GWh),Investment (Million $),CO2 Reduction (Tons),Jobs Created,Efficiency (%),Investment_per_GWh,CO2_per_Job
0,4,4,2022,-3.704824e-16,0.000000,0.959267,-1.447837,1.422105,-0.207758,2.072832
1,2,1,2023,-1.101199e+00,-1.387570,0.566660,0.810678,-0.181170,-0.489263,-0.311459
2,1,2,2021,7.196798e-02,-0.993420,-0.866974,-0.001050,1.021286,-0.815562,-0.675786
3,3,0,2020,1.432516e+00,1.226692,0.249562,-1.209094,-1.383626,-0.277616,0.889817
4,0,1,2020,-9.073008e-01,-0.140768,0.568124,-1.385764,1.502268,0.409919,1.517575
...,...,...,...,...,...,...,...,...,...,...
95,2,1,2022,-1.378197e+00,0.888849,0.815360,-1.505136,-0.982807,2.149734,2.083300
96,1,1,2024,8.019388e-01,-1.274956,-0.968629,0.863202,1.101450,-1.164056,-0.887360
97,4,4,2024,-3.793756e-01,-0.937113,1.642617,-1.648382,1.341941,-0.584839,3.596752
98,0,0,2023,-2.441354e-01,1.653018,-1.381526,0.175620,-1.543953,0.992957,-0.952796


In [23]:
# Final Dataset
# ============================================================

print(df.info())

print(df.describe())

print("Assignment Completed Successfully")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 10 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Energy Source            100 non-null    int64  
 1   Region                   100 non-null    int64  
 2   Year                     100 non-null    int64  
 3   Energy Production (GWh)  100 non-null    float64
 4   Investment (Million $)   100 non-null    float64
 5   CO2 Reduction (Tons)     100 non-null    float64
 6   Jobs Created             100 non-null    float64
 7   Efficiency (%)           100 non-null    float64
 8   Investment_per_GWh       100 non-null    float64
 9   CO2_per_Job              100 non-null    float64
dtypes: float64(7), int64(3)
memory usage: 7.9 KB
None
       Energy Source      Region         Year  Energy Production (GWh)  \
count     100.000000  100.000000   100.000000             1.000000e+02   
mean        2.060000    1.970000  2022.170000

In [ ]:
#-----Generalizations------#

# 1. Solar and Wind produce higher renewable energy than Biomass.

# 2. Higher investment generally increases energy production.

# 3. Regions with higher efficiency produce more electricity.

# 4. CO2 reduction increases as renewable energy production increases.

# 5. Energy production and jobs created show a positive relationship.

# 6. Efficiency is moderately related to CO2 reduction.

# 7. Investment and CO2 reduction have a positive correlation.

# 8. Hydropower usually has the highest energy production.

# 9. Biomass has comparatively lower efficiency.

# 10. Renewable energy projects generate employment while reducing pollution.

In [ ]:
#------Why Mean Imputation?------#

# Numerical values are continuous.
# Mean keeps the overall average unchanged.
# It is simple and suitable when few values are missing.

In [ ]:
#--------Why Feature Engineering?-------#

# Investment_per_GWh shows project cost efficiency.
# CO2_per_Job measures environmental impact created per employee.
# These new features provide additional insights beyond the original dataset.


In [ ]:
#------Why Label Encoding?-----#

# Converts categorical text into numbers.
# Useful for tree-based machine learning algorithms.

In [ ]:
#------Why One-Hot Encoding?------#

# Prevents algorithms from assuming an order between categories.
# Suitable for nominal categorical variables like Region and Energy Source.

In [ ]:
#--------Why StandardScaler?-------#

# StandardScaler brings all numerical features to a similar scale.
# It improves the performance of algorithms based on distance or gradients,
# such as KNN, SVM, Logistic Regression, and Neural Networks.


In [24]:
#------Final Insights------#

# Renewable energy production increases with investment.

# Higher efficiency leads to greater energy generation.

# Projects with larger investments reduce more CO2.

# Renewable energy development creates employment.

# Feature engineering reveals cost efficiency and environmental efficiency.

# Scaling improves numerical consistency.

# Encoding converts categorical data into machine-readable format.


